This is a **CLEAN** but vivid version of codes (along with visualizations) 
for analyzing the experiment ***freight collaboration-chessboard*** results

# Required packages

In [ ]:
from dataclasses import dataclass
from enum import Enum
import os
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
from itertools import product
import matplotlib.pyplot as plt
import seaborn as sns
# Use repo-relative path so it works on other machines
notebook_dir = Path.cwd() / "python" / "test"
if notebook_dir.exists() and str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))
import matsim
from pointpats import PointPattern, PoissonPointProcess
from pointpats.distance_statistics import g, f, k, l, j
import matsim_output_reader
import metric_anls
import figure_plot 
import agg_anls
import spatial_anls
# --- reload the module to reflect any changes made during development ---
import importlib
importlib.reload(matsim_output_reader)
importlib.reload(metric_anls)
importlib.reload(figure_plot)
importlib.reload(agg_anls)
importlib.reload(spatial_anls)

# Configuration

In [ ]:
# Resolve analysis path relative to repo root
repo_root = notebook_dir.parents[1]  # .../matsim-libs-2024
anls_path = repo_root / "output" / "chessboardCarrierReceiverCollab"

class DepotLocation(Enum):
    INSIDE = 'center'
    OUTSIDE = 'left'

class ReceiverDistribution(Enum):
    DISPERSED = 'DISPERSED'
    CLUSTERED = 'CLUSTERED'
    RANDOM = 'FULLY_RANDOM'

ALLOCATION_FACTOR = 0.8
ALLOCATION_FACTOR_LIST = [x / 10 for x in range(1, 10)]  # 0.1 to 0.9 with step of 0.1
#--- Penalty ---#
PENALTY_LIST = [0, 0.0003, 0.0008, 0.0014, 0.0028, 0.0056,
                0.0098, 0.014, 0.0167, 0.0222, 0.028, 0.0333, 0.0417]
PENALTY_LIST_SCALE = [round(x * 3600) for x in PENALTY_LIST]  # scale to avoid float precision issues
PENALTY_LIST_SCALE[PENALTY_LIST.index(0.028)] = 100 
PENALTY_DICT = {k: v for k, v in zip(PENALTY_LIST, PENALTY_LIST_SCALE)}
#--- Penalty ---#

TOTAL_INSTANCES = 60


In [ ]:
# Configuration for batch processing
# INPUT_PATH = str(anls_path)
OUTPUT_PATH = str(repo_root / "data" / "freightChessboardRC" / "clean")
DISPERSED_OUTPUT_PATH = str(repo_root / "data" / "freightChessboardRC" / "cleanMoreDispersed")
MORE_AF_OUTPUT_PATH = str(repo_root / "data" / "freightChessboardRC" / "cleanMoreAF")
MORE_PEN_OUTPUT_PATH = str(repo_root / "data" / "freightChessboardRC" / "cleanMorePen")

# Define which scenarios to process
DEPOT_LOCATIONS = [DepotLocation.INSIDE.value, DepotLocation.OUTSIDE.value]
RECEIVER_DISTRIBUTIONS = [ReceiverDistribution.DISPERSED.value, ReceiverDistribution.CLUSTERED.value]
ORIGINAL_TW = (6, 7)  # Original time window in hours
LAST_ITER = 30  # Last iteration number

In [ ]:
#--- Generate keywords for each scenario ---#
# Example tuple: ("center", "dispersed", 0)
scenario_keywords = [
    (depot, receiver, penalty)
    for depot, receiver, penalty in product([DepotLocation.INSIDE.value, DepotLocation.OUTSIDE.value],
                                            [ReceiverDistribution.DISPERSED.value, ReceiverDistribution.CLUSTERED.value], 
                                            PENALTY_LIST)
]
scenario_keywords[:3]

In [ ]:
OUTPUT_FIG_PATH = repo_root / "data" / "freightChessboardRC" / "figures"
OUTPUT_FIG_PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
network_dir = repo_root / "data" / "freightChessboardRC" / "output_network.xml.gz"
network = matsim.read_network(network_dir)
network_links = network.links
network_nodes = network.nodes

In [ ]:
# Build network graph once for efficiency
network_graph = agg_anls.build_network_graph(network_links, network_nodes)
print(f"Network graph: {network_graph.number_of_nodes()} nodes, {network_graph.number_of_edges()} edges")

In [ ]:
full_network_gdf = spatial_anls.network_graph_to_gdf(network_graph)
full_network_gdf

In [ ]:
central_area_network_gdf = spatial_anls.network_graph_to_gdf(
    network_graph,
    boundary=[2000, 2000, 7000, 7000])
central_area_network_gdf

# Read all_instance_metric df

In [ ]:
all_metrics_df = pd.read_csv(os.path.join(OUTPUT_PATH, 'all_scenarios_metrics.csv.gz'), compression='gzip')
all_metrics_df

In [ ]:
'''Add more AF instances (af->(0.1-0.4)) '''
ins_more_af_df = pd.read_csv(os.path.join(MORE_AF_OUTPUT_PATH, 'all_scenarios_metrics.csv.gz'), compression='gzip') 
ins_more_af_df

In [ ]:
''' add more penalty instances (0.1-0.4) with the AF 0.8 '''
ins_more_pen_af80 = pd.read_csv(os.path.join(MORE_PEN_OUTPUT_PATH, 'all_scenarios_metrics.csv.gz'), compression='gzip')
ins_more_pen_af80

In [ ]:
all_metrics_df = pd.concat([all_metrics_df, ins_more_af_df, ins_more_pen_af80], ignore_index=True)
all_metrics_df

In [ ]:
all_metrics_df = all_metrics_df[~all_metrics_df['instance'].between(20,29)]
all_metrics_df

In [ ]:
# aggregate_shipment_metrics_from_clean_folder function
clean_data_path = repo_root / 'data/freightChessboardRC/clean'
shipment_metrics_df = agg_anls.aggregate_shipment_metrics_from_clean_folder(str(clean_data_path), verbose=True)
shipment_metrics_df

In [ ]:
shipment_metrics_df = shipment_metrics_df[~shipment_metrics_df['instance_id'].between(20, 29)]
shipment_metrics_df['diff_fleet_size'] = shipment_metrics_df['final_fleet_size'] - shipment_metrics_df['iter0_fleet_size']

In [ ]:
all_metrics_df = pd.merge(all_metrics_df, 
                          shipment_metrics_df[['instance_id', 'allocation_factor', 'depot_location', 'receiver_distribution', 'penalty','diff_fleet_size', 'final_fleet_size']],
                          left_on=['instance', 'allocation_factor', 'depot_location', 'receiver_distribution', 'penalty'], 
                          right_on=['instance_id', 'allocation_factor', 'depot_location', 'receiver_distribution', 'penalty'], 
                          how='left')
all_metrics_df

In [ ]:
all_metrics_df['penalty_std'] = all_metrics_df['penalty'].map(PENALTY_DICT)
all_metrics_df 

## Rectify the dispersed related records

In [ ]:
''' drop all records with receiver_distribution == 'DISPERSED' '''
all_metrics_df = all_metrics_df[all_metrics_df['receiver_distribution'] != 'DISPERSED'].reset_index(drop=True)
all_metrics_df.shape

In [ ]:
ins_dispersed_metrics_df = pd.read_csv(os.path.join(DISPERSED_OUTPUT_PATH, 'all_scenarios_metrics.csv.gz'), compression='gzip')
ins_dispersed_metrics_df

In [ ]:
CORRECT_DISPERSED_INS_LIST = ins_dispersed_metrics_df['instance'].unique().tolist()
CORRECT_DISPERSED_INS_LIST

In [ ]:
''' Concat the dispersed metrics to the main metrics DataFrame '''
all_metrics_df = pd.concat([all_metrics_df, ins_dispersed_metrics_df], ignore_index=True)
all_metrics_df.shape

In [ ]:
all_metrics_df = all_metrics_df[all_metrics_df['instance'].isin(CORRECT_DISPERSED_INS_LIST)].reset_index(drop=True)
all_metrics_df.shape

## Split to specific dfs

In [ ]:
all_metrics_df['collaboration_rate'] = all_metrics_df['num_collaborative_receivers'] / 10
all_metrics_df

In [ ]:
''' Random dfs '''
all_metrics_random_df = all_metrics_df[all_metrics_df['receiver_distribution'] == ReceiverDistribution.RANDOM.value]
all_metrics_center_random_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.INSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.RANDOM.value)]
all_metrics_outside_random_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.OUTSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.RANDOM.value)]


In [ ]:
all_metrics_center_clustered_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.INSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value)]
all_metrics_center_dispersed_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.INSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.DISPERSED.value)]
all_metrics_outside_clustered_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.OUTSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value)]
all_metrics_outside_dispersed_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.OUTSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.DISPERSED.value)]

In [ ]:
all_metrics_non_random_df = all_metrics_df[all_metrics_df['receiver_distribution'] != ReceiverDistribution.RANDOM.value]

# Random scenario analysis
1. To demonstrate the effectiveness of the freight collaboration
2. To point out the possible effects/impacts of the spatial distribution of receivers

## Penalty sweep
The `allocation_factor` is fixed as 0.6

### Collaboration rate (box plot)

In [ ]:
figure_plot.box_plot(
    data_list=all_metrics_random_df.query("allocation_factor == 0.6"),
    cat_col='penalty_std',
    col_name='collaboration_rate',
    box_colors=["#83A7BE"] * len(PENALTY_LIST),
    # box_colors=sns.color_palette("Blues", n_colors=len(PENALTY_LIST)),
    box_alpha=0.9,
    se_box_color_for_lines=True,
     # Display options
    flier_marker='X',
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c4596e",
    median_linewidth=1,
    show_mean=True,
    mean_size=5,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Collaboration rate',
    xlabel='Cost of collaboration (euros/hour)',
    #--- Violin plot ---
    show_violin=False,
    violin_alpha=0.3, 
    violin_width=0.9,
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(8, 5),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='collaboration_rate_by_penalty_af60_all_random_receivers.png',
)

In [ ]:
figure_plot.box_plot(
    data_list=all_metrics_center_random_df.query("allocation_factor == 0.8"),
    cat_col='penalty_std',
    col_name='collaboration_rate',
    box_colors=["#83A7BE"] * len(PENALTY_LIST),
    box_alpha=0.9,
    se_box_color_for_lines=True,
     # Display options
    flier_marker='X',
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c4596e",
    median_linewidth=1,
    show_mean=True,
    mean_size=5,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Collaboration rate',
    xlabel='Cost of collaboration (euros/hour)',
    #--- Violin plot ---
    show_violin=True,
    violin_alpha=0.3, 
    violin_width=0.9,
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(6, 4),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='collaboration_rate_by_penalty_af80_center_random_receivers.png',
)

In [ ]:
figure_plot.box_plot(
    data_list=all_metrics_outside_random_df.query("allocation_factor == 0.8"),
    cat_col='penalty_std',
    col_name='collaboration_rate',
    box_colors=["#83A7BE"] * len(PENALTY_LIST),
    box_alpha=0.9,
    se_box_color_for_lines=True,
    # Display options
    flier_marker='X',
    median_color="#c4596e",
    median_linewidth=1,
    show_mean=True,
    mean_size=5,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Collaboration rate',
    xlabel='Cost of collaboration (euros/hour)',
    # Show scatter points with jitter
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    #--- Violin plot ---
    show_violin=True,
    violin_alpha=0.3, 
    violin_width=0.9,
    #--- Grid ---
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(6, 4),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='collaboration_rate_by_penalty_af80_outside_random_receivers.png',
)

In [ ]:
metric_anls.compute_ashmans_d(
    df=all_metrics_center_random_df.query("penalty_std == 20"),
    col_name='collaboration_rate',
)

### Collaboration rate (histogram)
Only plotting those groups with large variance

#### All-af60-p20

In [ ]:
figure_plot.hist_plot(all_metrics_random_df[(all_metrics_random_df['allocation_factor'] == 0.6) & (all_metrics_random_df['penalty_std'] == 20)],
                      col1='collaboration_rate',
                      n_bins=8,
                      figure_size=(4,2.5),
                      ylabel='Density',
                      xlabel='Collaboration rate',
                      label_size=14,
                      hide_labels=False,
                      hide_legends=True,
                      alphas=(0.9, 0.9),
                      colors=("#8dadc3", '#f6bdb1'),
                      #---fitting---
                      fitting_method='kde',
                      kde_bw=0.3,
                      figure_folder=OUTPUT_FIG_PATH,  
                      filename='collab_rate_distribution_all_random_af60_p20euro.png'
                      )
metric_anls.compute_ashmans_d(
    df=all_metrics_random_df[(all_metrics_random_df['allocation_factor'] == 0.6) & (all_metrics_random_df['penalty_std'] == 20)],
    col_name='collaboration_rate',
)

#### All-af60-p35

In [ ]:
figure_plot.hist_plot(all_metrics_random_df[(all_metrics_random_df['allocation_factor'] == 0.6) & (all_metrics_random_df['penalty_std'] == 35)],
                      col1='collaboration_rate',
                      n_bins=8,
                      figure_size=(4,2.5),
                      ylabel='Density',
                      xlabel='Collaboration rate',
                      label_size=14,
                      hide_labels=False,
                      hide_legends=True,
                      alphas=(0.9, 0.9),
                      colors=("#8dadc3", '#f6bdb1'),
                      #---fitting---
                      fitting_method='gmm',
                      figure_folder=OUTPUT_FIG_PATH,  
                      filename='collab_rate_distribution_all_random_af60_p35euro.png'
                      )

#### All-af60-p50

In [ ]:
figure_plot.hist_plot(all_metrics_random_df[
    (all_metrics_random_df['allocation_factor'] == 0.6) & 
    (all_metrics_random_df['penalty_std'] == 50)
    ],
                      col1='collaboration_rate',
                      n_bins=5,
                      figure_size=(3.5,2.7),
                      ylabel='Density',
                      xlabel='Collaboration rate',
                      label_size=10,
                      hide_labels=False,
                      hide_legends=True,
                      alphas=(0.9, 0.9),
                      colors=("#8dadc3", '#f6bdb1'),
                      #---fitting---
                      fitting_method='kde',
                      figure_folder=OUTPUT_FIG_PATH,  
                      filename='collab_rate_distribution_all_random_af60_p50euro.png'
                      )

### Total cost savings

In [ ]:
figure_plot.box_plot(
    data_list=all_metrics_random_df.query("allocation_factor == 0.6"),
    cat_col='penalty_std',
    col_name='total_cost_savings',
    box_colors=["#83A7BE"] * len(PENALTY_LIST),
    # box_colors=sns.color_palette("Blues", n_colors=len(PENALTY_LIST)),
    box_alpha=0.9,
    se_box_color_for_lines=True,
     # Display options
    flier_marker='X',
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c4596e",
    median_linewidth=1,
    show_mean=True,
    mean_size=5,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Total cost savings',
    xlabel='Cost of collaboration (euros/hour)',
    #--- Violin plot ---
    show_violin=False,
    violin_alpha=0.3, 
    violin_width=0.9,
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(4, 3),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='total_cost_savings_by_penalty_af60_all_random_receivers.png',
)

In [ ]:
all_metrics_random_df.query("allocation_factor == 0.8").plot.scatter(x='penalty_std', y='total_cost_savings')

### fleet size

In [ ]:
_df = all_metrics_random_df.query("allocation_factor == 0.6")
_df['diff_fleet_size'] = _df['diff_fleet_size'].map(lambda x: 0 if x >=1 else x)  # Map all values >= 1 to 0, keep others unchanged
_df['diff_fleet_size'] = _df['diff_fleet_size'] * -1

_colors = figure_plot.generate_smooth_colors(
    input_colors=["#abd1e9", "#003b64"],
    n_req_colors=4
)
figure_plot.stacked_proportion_plot(_df,
                                    figure_size=(12,4),
                                    dpi=350,
                                     bin_col='penalty_std',
                                     value_col='final_fleet_size',
                                    # n_bins=5,
                                     bin_method='unique',
                                    #  custom_bins=[0.4, 0.6, 0.8, 1.0],
                                    #  colors=['#757575',"#abd1e9", "#458CBC", 
                                    #         #  "#005681DF",
                                    #          "#003b64"],
                                     colors=_colors,
                                     xlabel='Cost of collaboration (euros/hour)',
                                     ylabel='Proportion (%)',
                                     label_size=14,
                                     show_counts=False,
                                     count_fontsize=12,
                                     percentage_fontsize=12,
                                     # Legend settings
                                     legend_bbox=(0.5, 1.4),
                                     legend_ncol=3,
                                     legend_title='Fleet size',
                                     show_legend=False,
                                     # Output
                                     figure_folder=OUTPUT_FIG_PATH,
                                     filename='all_random_af60_stacked_proportion_fleet_size_vs_penalty.png'
                                     )

### Scores

In [ ]:
_df = all_metrics_random_df.query("allocation_factor == 0.8").groupby('penalty_std')['final_carrier_score'].agg(['mean', 'std', 'count'])
_df.plot.line( y='mean')

### VKT

In [ ]:
figure_plot.box_plot(
    data_list=all_metrics_random_df.query("allocation_factor == 0.6"),
    cat_col='penalty_std',
    col_name='VKT_km',
    box_colors=["#83A7BE"] * len(PENALTY_LIST),
    # box_colors=sns.color_palette("Blues", n_colors=len(PENALTY_LIST)),
    box_alpha=0.9,
    se_box_color_for_lines=True,
     # Display options
    flier_marker='X',
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c4596e",
    median_linewidth=1,
    show_mean=True,
    mean_size=5,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='VKT',
    xlabel='Cost of collaboration (euros/hour)',
    #--- Violin plot ---
    show_violin=False,
    violin_alpha=0.3, 
    violin_width=0.9,
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(4, 3),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='VKT_by_penalty_af60_all_random_receivers.png',
)

### TKT

In [ ]:
_tkt_df = all_metrics_random_df.query("allocation_factor == 0.6")
_tkt_df['TKT_tonkm'] = _tkt_df['TKT_tonkm'] / 1000  # Scale to thousand ton-km

figure_plot.box_plot(
    data_list=_tkt_df,
    cat_col='penalty_std',
    col_name='TKT_tonkm',
    box_colors=["#83A7BE"] * len(PENALTY_LIST),
    # box_colors=sns.color_palette("Blues", n_colors=len(PENALTY_LIST)),
    box_alpha=0.9,
    se_box_color_for_lines=True,
     # Display options
    flier_marker='X',
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c4596e",
    median_linewidth=1,
    show_mean=True,
    mean_size=5,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Ton-km travelled',
    xlabel='Cost of collaboration (euros/hour)',
    #--- Violin plot ---
    show_violin=False,
    violin_alpha=0.3, 
    violin_width=0.9,
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(4, 3),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='TKT_by_penalty_af60_all_random_receivers.png',
)

## Allocation sweep

### Collaboration rate

In [ ]:
figure_plot.box_plot(
    data_list=all_metrics_random_df.query("penalty_std == 20"),
    cat_col='allocation_factor',
    col_name='collaboration_rate',
    box_colors=["#83A7BE"] * len(PENALTY_LIST),
    # box_colors=sns.color_palette("Blues", n_colors=len(PENALTY_LIST)),
    box_alpha=0.9,
    se_box_color_for_lines=True,
     # Display options
    flier_marker='X',
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c4596e",
    median_linewidth=1,
    show_mean=True,
    mean_size=5,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Collaboration rate',
    xlabel='Allocation factor',
    #--- Violin plot ---
    show_violin=True,
    violin_alpha=0.3, 
    violin_width=0.9,
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(6, 4),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    # filename='collaboration_rate_by_penalty_af80_all_random_receivers.png',
)

## Joint-af-pen

In [ ]:
all_metrics_random_df[all_metrics_random_df['penalty_std']>100]

In [ ]:
joint_af_pen_df = all_metrics_random_df.copy(deep=True)
joint_af_pen_df = joint_af_pen_df.query("penalty_std >0 and penalty_std <=100")


### Heatmap for collab

In [ ]:
_colors = figure_plot.generate_smooth_colors(
    input_colors=["#F7F2F2","#702F35"],
    n_req_colors=10
)
figure_plot.heatmap_plot(
    joint_af_pen_df,
    x_col='penalty_std',
    y_col='allocation_factor',
    value_col='collaboration_rate',
    agg_method='mean',
    cmap=_colors,
    #--- Annotation options ---
    annot=False,
    annot_size=12,
    # annot_color='black',
    annot_fmt='.2f',
    xlabel='Cost of collaboration (euros/hour)',
    ylabel='Allocation factor',
    label_size=14,
    #--- Output ---
    cbar=False,
    figure_size=(6, 6),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='heatmap_collaboration_rate_by_penalty_and_allocation_factor_random_receivers.png',
    # tick_label_map=PENALTY_DICT,
)

### Change rate lines

In [ ]:
collab_value_df = all_metrics_random_df.query("penalty_std <=100").groupby(['allocation_factor', 'penalty_std'])['collaboration_rate'].mean().unstack()
collab_value_df.fillna(1, inplace=True)  # Fill pen-0 NaN values with 1
collab_value_df

In [ ]:
change_value_df = collab_value_df.shift(1, axis=1) - collab_value_df
change_value_df = change_value_df.iloc[:, 1:]  # Remove the first column which will be NaN after shift
# # change it to percentage change
# change_value_df = change_value_df.apply(lambda x: x / collab_value_df[x.name] * 100)
_list = [0.2, 0.4, 0.6, 0.8]
# _list = [0.1, 0.3, 0.5, 0.7, 0.9]
change_value_df = change_value_df[change_value_df.index.isin(_list)]
change_value_df

In [ ]:
''' Plot the lines for each allocation factor-penalty change'''
_colors= figure_plot.generate_smooth_colors(
    input_colors=["#C7B7F8","#292252"],  # #463A8B
    n_req_colors=change_value_df.shape[0]
)
figure_plot.plot_change_rate(
    pivot_df=change_value_df,
    xlabel='Cost of collaboration (euros/hour)',
    ylabel='Collaboration rate drop',
    label_size=14,
    legend_title='Allocation factor',
    legend_frameon=False,
    colors=_colors,
    # alpha=0.6,
     #--- Output ---
    show_legend=True,
    figure_size=(5, 3),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='change_rate_collaboration_rate_by_penalty_random_receivers.png',
)


In [ ]:
sns.color_palette("tab20c")

In [ ]:
collab_value_df = all_metrics_random_df.query("penalty_std <= 100").groupby(['penalty_std', 'allocation_factor',])['collaboration_rate'].mean().unstack()
collab_value_df = collab_value_df.apply(lambda x: round(x,2))
collab_value_df[0.0] = 0
collab_value_df = collab_value_df.sort_index(axis=1) 
change_value_df = collab_value_df - collab_value_df.shift(1, axis=1)
change_value_df = change_value_df.iloc[:, 1:] 
change_value_df = change_value_df.applymap(lambda x: 0 if x <= 0 else x)  # Map all positive changes to 0, keep negative changes unchanged
change_value_df = change_value_df[change_value_df.index.to_series().between(10, 50)]

change_value_df

In [ ]:
_colors = figure_plot.generate_smooth_colors(
    input_colors=["#b6dbe3","#1D483F"],
    n_req_colors=change_value_df.shape[1]
)
_colors = ["#badad3", '#97b0aa', "#5f7570", '#1D483F']
figure_plot.plot_change_rate(
    # data_df=all_metrics_random_df[all_metrics_random_df['penalty_std'].between(10, 60)],
    # group_col='penalty_std',
    # x_col='allocation_factor',
    # value_col='collaboration_rate',
    # agg_method='mean',
    pivot_df=change_value_df,
    xlabel='Allocation factor',
    ylabel='Collaboration rate rise',
    label_size=14,
    legend_title='Cost of collab.',
    legend_frameon=False,
    colors=_colors,
    #--- Output ---
    # show_legend=False,
    show_legend=True,
    figure_size=(5, 3),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='change_rate_collaboration_rate_by_allocation_factor_random_receivers.png',
)

### Keep collaboration rate lines

In [ ]:
''' Full collaborate '''
_colors = figure_plot.generate_smooth_colors(
    input_colors=["#F7F2F2","#702F35"],
    n_req_colors=10
)
figure_plot.heatmap_plot(
    joint_af_pen_df,
    x_col='penalty_std',
    y_col='allocation_factor',
    value_col='collaboration_rate',
    agg_method='mean',
    annot=False,
    cmap=_colors,
    # --- Highlight cells with value >= 0.8 ---
    cap_value=0.91,
    cap_edgecolor="#C72323",       # border colour
    cap_linewidth=2,         # border width
    cap_linestyle='--',         # '-', '--', ':', etc.
    cap_compare='>=',          # '>=', '>', '<=', '<', '==', '!='
    # --- cap line ---
    cap_outer_only=True,       # draw only the outer boundary of contiguous highlighted regions
    cap_fill=True,           # optional translucent fill
    cap_fill_color="#F08787",
    cap_fill_alpha=0.4,
    #---output---
    cbar=False,
    xlabel='Cost of collaboration (euros/hour)',
    ylabel='Allocation factor',
    label_size=14,
    figure_size=(4, 3.5),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='heatmap_collaboration_rate_by_penalty_and_allocation_factor_random_receivers_with_highlight_greater1.png',

)

In [ ]:
_colors = figure_plot.generate_smooth_colors(
    input_colors=["#F7F2F2","#702F35"],
    n_req_colors=10
)
figure_plot.heatmap_plot(
    joint_af_pen_df,
    x_col='penalty_std',
    y_col='allocation_factor',
    value_col='collaboration_rate',
    agg_method='mean',
    annot=False,
    cmap=_colors,
    # --- Highlight cells with value >= 0.8 ---
    cap_value=0.69,
    cap_edgecolor="#C72323",       # border colour
    cap_linewidth=2,         # border width
    cap_linestyle='--',         # '-', '--', ':', etc.
    cap_compare='>=',          # '>=', '>', '<=', '<', '==', '!='
    # --- cap line ---
    cap_outer_only=True,       # draw only the outer boundary of contiguous highlighted regions
    cap_fill=True,           # optional translucent fill
    cap_fill_color="#F08787",
    cap_fill_alpha=0.4,
    #---output---
    cbar=False,
    xlabel='Cost of collaboration (euros/hour)',
    ylabel='Allocation factor',
    label_size=14,
    figure_size=(4, 3.5),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='heatmap_collaboration_rate_by_penalty_and_allocation_factor_random_receivers_with_highlight_greater70.png',

)

In [ ]:
_colors = figure_plot.generate_smooth_colors(
    input_colors=["#F7F2F2","#702F35"],
    n_req_colors=10
)
figure_plot.heatmap_plot(
    joint_af_pen_df,
    x_col='penalty_std',
    y_col='allocation_factor',
    value_col='collaboration_rate',
    agg_method='mean',
    annot=False,
    cmap=_colors,
    # --- Highlight cells with value >= 0.5 ---
    cap_value=0.50,
    cap_edgecolor="#C72323",       # border colour
    cap_linewidth=2,         # border width
    cap_linestyle='--',         # '-', '--', ':', etc.
    cap_compare='>=',          # '>=', '>', '<=', '<', '==', '!='
    # --- cap line ---
    cap_outer_only=True,       # draw only the outer boundary of contiguous highlighted regions
    cap_fill=True,           # optional translucent fill
    cap_fill_color="#F08787",
    cap_fill_alpha=0.4,
    #---output---
    cbar=False,
    xlabel='Cost of collaboration (euros/hour)',
    ylabel='Allocation factor',
    label_size=14,
    figure_size=(4, 3.5),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='heatmap_collaboration_rate_by_penalty_and_allocation_factor_random_receivers_with_highlight_greater50.png',

)

## Analyse random scenario before-and-after operation

In [ ]:
all_metrics_random_20euro_af60_df = all_metrics_random_df[(all_metrics_random_df['penalty'] == 0.0056) & (all_metrics_random_df['allocation_factor'] == 0.6)]
all_metrics_random_20euro_af60_df

In [ ]:
print("the mean collaboration rate for 20 euro penalty is:", all_metrics_random_20euro_af60_df['collaboration_rate'].mean())
print("the max collaboration rate for 20 euro penalty is:", all_metrics_random_20euro_af60_df['collaboration_rate'].max())
print("the min collaboration rate for 20 euro penalty is:", all_metrics_random_20euro_af60_df['collaboration_rate'].min())

### VKT

In [ ]:
_vkt_df = all_metrics_outside_random_20euro_af80_df[all_metrics_outside_random_20euro_af80_df['instance'].between(0, 100)]
figure_plot.hist_plot(_vkt_df,
                      col1='VKT_km',
                      col2='iter0_VKT_km',
                      n_bins=10,
                      figure_size=(3.5,2.7),
                      ylabel = 'Density',
                      xlabel = 'VKT (km)',
                      label_size=14,
                      hide_labels=False,
                      hide_legends=True,
                      alphas=(0.9, 0.9),
                      colors=("#8dadc3", '#f6bdb1'),
                      figure_folder=OUTPUT_FIG_PATH,
                      filename='vkt_distribution_outside_random_20euro_penalty'
                      )
print("the mean reduction in VKT for 20 euro penalty is:",
       (_vkt_df['iter0_VKT_km'].mean() - _vkt_df['VKT_km'].mean()))

In [ ]:
_vkt_df = all_metrics_random_20euro_af60_df.copy(deep=True)
_vkt_df['vkt_reduction'] = _vkt_df['iter0_VKT_km'] - _vkt_df['VKT_km']

In [ ]:
figure_plot.scatter_regression_plot(_vkt_df,
                                    x_col='collaboration_rate',
                                    y_col='vkt_reduction',
                                    figure_size=(3.3,2.8),
                                    xlabel='Collaboration rate',
                                    ylabel='VKT reduction',
                                    label_size=14,
                                    annotation_fontsize=12,
                                    # title='Scatter plot with regression line: VKT vs Collaboration Rate',
                                    show_corr=True,
                                    add_regression=True,
                                    show_equation=False,
                                    show_r2=True,
                                    scatter_color='#458CBC',
                                    scatter_alpha=0.9,
                                    reg_color='#0b2c60',
                                    # Output
                                    figure_folder=OUTPUT_FIG_PATH,
                                    filename='scatter_regression_vkt_reduction_vs_collab_rate_all_random_20euro_af60.png'
                                    )

### VTT

In [ ]:
vtt_minutes_df = all_metrics_outside_random_20euro_af80_df.copy()
vtt_minutes_df[['VTT_seconds', 'iter0_VTT_seconds']] = (
    vtt_minutes_df[['VTT_seconds', 'iter0_VTT_seconds']] / 60
)
_vtt_minutes_df = vtt_minutes_df[vtt_minutes_df['instance'].between(0,100)]
figure_plot.hist_plot(_vtt_minutes_df,
                      col1='VTT_seconds',
                      col2='iter0_VTT_seconds',
                      n_bins=10,
                      figure_size=(3.5,2.7),
                      ylabel='Density',
                      xlabel='Travel time (min)',
                      label_size=14,
                      hide_labels=False,
                      hide_legends=True,
                      alphas=(0.9, 0.9),
                        colors=("#8dadc3", '#f6bdb1'),
                        figure_folder=OUTPUT_FIG_PATH,
                        filename='vtt_distribution_outside_random_20euro_penalty'
                      )
print("the mean reduction of VTT is: {}".format(
    -(_vtt_minutes_df['VTT_seconds'].mean() 
        - _vtt_minutes_df['iter0_VTT_seconds'].mean())) 
        )

### Ton-km travelled

In [ ]:
tkt_ton_df = all_metrics_outside_random_20euro_af80_df.copy()
tkt_ton_df[['TKT_tonkm', 'iter0_TKT_tonkm']] = (
    tkt_ton_df[['TKT_tonkm', 'iter0_TKT_tonkm']] / 1000
)
_tkt_ton_df = tkt_ton_df[tkt_ton_df['instance'].between(0,100)]
figure_plot.hist_plot(_tkt_ton_df,
                        col1='TKT_tonkm',
                        col2='iter0_TKT_tonkm',
                        n_bins=9,
                        figure_size=(3.3,2.8),
                        ylabel='Density',
                        xlabel='Ton-km travelled',
                        hide_labels=False,
                        hide_legends=True,
                        alphas=(0.9, 0.9),
                        colors=("#8dadc3", '#f6bdb1'),
                        label_size=14,
                        figure_folder=OUTPUT_FIG_PATH,
                        filename='tkt_distribution_outside_random_20euro_penalty'
                        )
print("the mean increase of TKT is: {}".format(
    (_tkt_ton_df['TKT_tonkm'].mean()
        - _tkt_ton_df['iter0_TKT_tonkm'].mean())) 
        )


In [ ]:
_tkt_df = all_metrics_random_20euro_af60_df.copy(deep=True)
_tkt_df['TKT_rise'] = _tkt_df['TKT_tonkm'] - _tkt_df['iter0_TKT_tonkm']
_tkt_df['TKT_rise'] = _tkt_df['TKT_rise'] / 1000
_tkt_df = _tkt_df[_tkt_df['TKT_rise'] >= 0]
_tkt_df

In [ ]:
figure_plot.scatter_regression_plot(_tkt_df,
                                    x_col='collaboration_rate',
                                    y_col='TKT_rise',
                                    figure_size=(3.3,2.8),
                                    xlabel='Collaboration rate',
                                    ylabel='TKT rise',
                                    label_size=14,
                                    annotation_fontsize=12,
                                    # title='Scatter plot with regression line: TKT Rise vs Collaboration Rate',
                                    show_corr=True,
                                    add_regression=True,
                                    show_equation=False,
                                    show_r2=True,
                                    scatter_color='#458CBC',
                                    scatter_alpha=0.9,
                                    reg_color='#0b2c60',
                                    # Output
                                    figure_folder=OUTPUT_FIG_PATH,
                                    filename='scatter_regression_tkt_rise_vs_collab_rate_all_random_20euro_af60.png'
                                    )

### Joint plot of collaboration rate, carrier score, receiver score and fleet size.


In [ ]:
_score_df.query("total_receiver_scores >= -825")

In [ ]:
_score_df = all_metrics_random_20euro_af60_df[all_metrics_random_20euro_af60_df['instance'].between(0,100)]
_score_df.query("final_carrier_score >= 0 and total_receiver_scores >= -1000", inplace=True)
_score_df['iter0_total_receiver_scores'] = -1000

_colors = figure_plot.generate_smooth_colors(
    input_colors=["#abd1e9","#0b2c60"],
    n_req_colors=11
)

figure_plot.joint_scatter_plot(
    _score_df,
    x_col='final_carrier_score',
    y_col='total_receiver_scores',
    n_bins=10,
    figure_size=(6, 6),
    marginal_type='both',  # 同时显示 histogram 和 KDE
    marginal_label_size=14,
    xlabel='Carrier score',
    ylabel='Receiver score',
    label_size=16,
    show_corr=False,
    # scatter_color="#8dadc3",
    scatter_alpha=0.9,
    hist_color="#4F81A3",
    kde_linestyle='--',
    kde_alpha=0.9,
    ## the second scatter plot
    # data_df2=_score_df,
    # x_col2='iter0_carrier_score',
    # y_col2='iter0_total_receiver_scores',
    # scatter2_marker='X',
    # scatter2_color='#f6bdb1',
    # scatter2_edgecolor="#f09785",
    # scatter2_alpha=0.9,
    # scatter2_size=90,
    ## Group the main scatter points by collaboration rate
    size_group_col='collaboration_rate',
    size_min=30,
    size_max=300,
    show_size_legend=False,
    # size_legend_loc='lower right',
    ## Group the main scatter points by fleet size
    color_group_col='collaboration_rate',
    # color_group_col='diff_fleet_size',
    cmap_color_list=_colors,
    show_color_legend=False,
    ## Output
    # add_regression=True,
    figure_folder=OUTPUT_FIG_PATH,
    # filename='joint_RC_scores_with_collab_rate_all_random_20euro_af60.png',
)


### Fleet size

In [ ]:
pd.cut(_score_df['collaboration_rate'], bins=[0, 0.4, 0.6, 0.8, 1.0], retbins=True, include_lowest=True)

In [ ]:
_score_df

In [ ]:
# _colors = figure_plot.generate_smooth_colors(
#     input_colors=["#abd1e9", "#0b2c60"],
#     n_req_colors=11
# )
figure_plot.stacked_proportion_plot(_score_df,
                                    figure_size=(3.5,3),
                                     bin_col='collaboration_rate',
                                     value_col='diff_fleet_size',
                                     bin_method='custom',
                                     custom_bins=[0, 0.3, 0.6, 1.0],
                                     xtick_labels=['Low \n(0-0.3)', 'Medium \n(0.4-0.6)', 'High \n(0.7-1.0)'],
                                     xtick_rotation=0,
                                     xtick_ha='center',
                                     xtick_fontsize=12,
                                     colors=["#b6b8b7", "#abd1e9", "#458CBC", "#0b2c60"][::-1],
                                     xlabel='Collaboration rate',
                                     ylabel='Proportion (%)',
                                     label_size=14,
                                     show_counts=True,
                                     count_fontsize=12,
                                     percentage_fontsize=14,
                                     # Legend settings
                                     legend_bbox=(0.5, 1.4),
                                     legend_ncol=3,
                                     legend_title='Fleet size difference',
                                     show_legend=False,
                                     # Output
                                     figure_folder=OUTPUT_FIG_PATH,
                                     filename='stacked_proportion_fleet_size_diff_vs_collab_all_random_af60_pen20.png'
                                     )

In [ ]:
figure_plot.scatter_regression_plot(all_metrics_random_20euro_af60_df,
                                    x_col='collaboration_rate',
                                    y_col='diff_fleet_size',
                                    figure_size=(4,3),
                                    xlabel='Collaboration rate',
                                    ylabel='Fleet size reduction',
                                    label_size=14,
                                    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
                                    show_corr=True,
                                    add_regression=True,
                                    show_equation=False,
                                    show_r2=False,
                                    scatter_color='#458CBC',
                                    scatter_alpha=0.9,
                                    reg_color='#0b2c60',
                                    # Output
                                    figure_folder=OUTPUT_FIG_PATH,
                                    filename='scatter_regression_fleet_size_reduction_vs_collab_rate_all_random_20euro_af60.png'
                                    )

### Total cost savings
Plot the correlations between the collaboration rate and the total cost savings

In [ ]:
figure_plot.scatter_regression_plot(all_metrics_random_20euro_af60_df,
                                    x_col='collaboration_rate',
                                    y_col='total_cost_savings',
                                    figure_size=(3.3,2.8),
                                    xlabel='Collaboration rate',
                                    ylabel='Total cost savings',
                                    label_size=14,
                                    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
                                    show_corr=True,
                                    add_regression=True,
                                    show_equation=False,
                                    show_r2=True,
                                    annotation_fontsize=12,
                                    scatter_color='#458CBC',
                                    scatter_alpha=0.9,
                                    reg_color='#0b2c60',
                                    # Output
                                    figure_folder=OUTPUT_FIG_PATH,
                                    filename='scatter_regression_cost_savings_vs_collab_rate_all_random_20euro_af60.png'
                                    )

## Compare specific low-run and high-run random scenarios

In [ ]:
all_metrics_random_20euro_af60_df.query("collaboration_rate == 0")

In [ ]:
all_metrics_random_20euro_af60_df.query("collaboration_rate == 1")

In [ ]:
_low_collab_ins_id = 15 # or 9
_high_collab_ins_id = 18
_second_high_collab_ins_id = 49

anls_af = 0.6
# _medium_collab_ins_id = 41
# specific_run_list = [_low_collab_ins_id, _high_collab_ins_id, _medium_collab_ins_id]

In [ ]:
low_collab_ins_receiver_df = gpd.read_file(os.path.join(
    OUTPUT_PATH, 
    f'{DepotLocation.INSIDE.value}-{ReceiverDistribution.CLUSTERED.value}-af{anls_af:.02f}-p{0.0056}-i{_low_collab_ins_id:02d}',
    'geo_data.geojson'))
high_collab_ins_receiver_df = gpd.read_file(os.path.join(
    OUTPUT_PATH, 
    f'{DepotLocation.OUTSIDE.value}-{ReceiverDistribution.RANDOM.value}-af{anls_af:.02f}-p{0.0056}-i{_high_collab_ins_id:02d}',
    'geo_data.geojson'))    
second_high_collab_ins_receiver_df = gpd.read_file(os.path.join(
    OUTPUT_PATH,
    f'{DepotLocation.OUTSIDE.value}-{ReceiverDistribution.RANDOM.value}-af{anls_af:.02f}-p{0.0056}-i{_second_high_collab_ins_id:02d}',
    'geo_data.geojson'))


### Plot spatial distribution

#### High

In [ ]:
''' Plot receiver locations at link midpoints for high collaboration scenario'''
figure_plot.network_locations_plot(
    network=network,
    locations_gdf=high_collab_ins_receiver_df,
    # Links settings
    show_links=True,
    show_nodes=True,
    use_arrows=True,
    link_color="#878A8B",
    link_alpha=0.6,
    link_linewidth=0.6,
    arrow_scale=12,
    arrow_shrink=5,
    # Nodes settings
    node_size=15,
    node_color="#A7A7A7",
    node_alpha=0.8,
    # 使用link中点定位 (新功能)
    use_link_midpoint=True,
    link_id_col='link_id',
    # Central region
    show_central_region=True,
    central_region_bounds=(2000, 2000, 5000, 5000),
    central_region_label='Central Region (2000-7000)',
    #---X/Y labels---
    xlabel='X coordinate (m)',
    ylabel='Y coordinate (m)',
    label_size=14,

    # Axis limits
    xlim=(-500, 9500),
    ylim=(-500, 9500),
    #---Title and style---
    # title='Receiver Locations at Link Midpoints (High Collaboration)',
    figure_size=(4, 4),
    dpi=350,
    show_grid=True,
    show_legend=False,
    #---Output---
    figure_folder=OUTPUT_FIG_PATH,
    filename='receiver_locations_high_collab_outside_random_af60_20euro.png'
)

#### 2nd High

In [ ]:
''' Plot receiver locations at link midpoints for medium collaboration scenario'''

figure_plot.network_locations_plot(
    network=network,
    locations_gdf=second_high_collab_ins_receiver_df,
    # Links settings
    show_links=True,
    show_nodes=True,
    use_arrows=True,
    link_color="#878A8B",
    link_alpha=0.6,
    link_linewidth=0.6,
    arrow_scale=12,
    arrow_shrink=5,
    # Nodes settings
    node_size=15,
    node_color="#A7A7A7",
    node_alpha=0.8,
    # 使用link中点定位 (新功能)
    use_link_midpoint=True,
    link_id_col='link_id',
    # Central region
    show_central_region=True,
    central_region_bounds=(2000, 2000, 5000, 5000),
    central_region_label='Central Region (2000-7000)',
    #---X/Y labels---
    xlabel='X coordinate (m)',
    ylabel='Y coordinate (m)',
    label_size=14,

    # Axis limits
    xlim=(-500, 9500),
    ylim=(-500, 9500),
    #---Title and style---
    # title='Receiver Locations at Link Midpoints (High Collaboration)',
    figure_size=(4, 4),
    dpi=350,
    show_grid=True,
    show_legend=False,
    #---Output---
    figure_folder=OUTPUT_FIG_PATH,
    filename='receiver_locations_second_high_collab_random_af60_20euro.png'
)

#### Low

In [ ]:
''' Plot receiver locations at link midpoints for low collaboration scenario '''
figure_plot.network_locations_plot(
    network=network,
    locations_gdf=low_collab_ins_receiver_df,
    # Links settings
    show_links=True,
    show_nodes=True,
    use_arrows=True,
    link_color="#878A8B",
    link_alpha=0.6,
    link_linewidth=0.6,
    arrow_scale=12,
    arrow_shrink=5,
    # Nodes settings
    node_size=15,
    node_color="#A7A7A7",
    node_alpha=0.8,
    # 使用link中点定位 (新功能)
    use_link_midpoint=True,
    link_id_col='link_id',
    # Central region
    show_central_region=True,
    central_region_bounds=(2000, 2000, 5000, 5000),
    central_region_label='Central Region (2000-7000)',
    #---X/Y labels---
    xlabel='X coordinate (m)',
    ylabel='Y coordinate (m)',
    label_size=14,

    # Axis limits
    xlim=(-500, 9500),
    ylim=(-500, 9500),
    #---Title and style---
    # title='Receiver Locations at Link Midpoints (Low Collaboration)',
    figure_size=(4, 4),
    dpi=350,
    show_grid=True,
    show_legend=False,
    #---Output---
    figure_folder=OUTPUT_FIG_PATH,
    filename='receiver_locations_low_collab_random_af60_20euro.png'
)

### Agg statistics

In [ ]:
com_df.plot.scatter(x='collaboration_rate', y='mean_receiver_dist_to_depot_euclidean_km')


In [ ]:
com_df.plot.scatter(x='collaboration_rate', y='clustering_index_euclidean_km')



In [ ]:
com_df_collab_40_50 = com_df[
    (com_df['collaboration_rate'].between(0.4, 0.6))
]
com_df_collab_60_80 = com_df[
    (com_df['collaboration_rate'].between(0.7, 0.8))
]
com_df_collab_90_100 = com_df[
    (com_df['collaboration_rate'].between(0.7, 1.0))
]

In [ ]:
## Print the mean receiver to depot euclidean distance for each subgroup
print("the mean receiver to depot euclidean distance for subgroup 40-50% is:", com_df_collab_40_50['mean_receiver_dist_to_depot_euclidean_km'].mean())
print("the mean receiver to depot euclidean distance for subgroup 60-80% is:", com_df_collab_60_80['mean_receiver_dist_to_depot_euclidean_km'].mean())
print("the mean receiver to depot euclidean distance for subgroup 90-100% is:", com_df_collab_90_100['mean_receiver_dist_to_depot_euclidean_km'].mean())

print()
## print the mean receiver to depot road distance for each subgroup
print("the mean receiver to depot road distance for subgroup 40-50% is:", com_df_collab_40_50['mean_receiver_dist_to_depot_network_km'].mean())
print("the mean receiver to depot road distance for subgroup 60-80% is:", com_df_collab_60_80['mean_receiver_dist_to_depot_network_km'].mean())
print("the mean receiver to depot road distance for subgroup 90-100% is:", com_df_collab_90_100['mean_receiver_dist_to_depot_network_km'].mean())

print()
## Print the mean clustering_index_euclidean_km for each subgroup
print("the mean clustering_index_euclidean_km for subgroup 40-50% is:", com_df_collab_40_50['clustering_index_euclidean_km'].mean())
print("the mean clustering_index_euclidean_km for subgroup 60-80% is:", com_df_collab_60_80['clustering_index_euclidean_km'].mean())
print("the mean clustering_index_euclidean_km for subgroup 90-100% is:", com_df_collab_90_100['clustering_index_euclidean_km'].mean())

print()
### Print the mean clustering_index_network_km for each subgroup
print("the mean clustering_index_network_km for subgroup 40-50% is:", com_df_collab_40_50['clustering_index_network_km'].mean())
print("the mean clustering_index_network_km for subgroup 60-80% is:", com_df_collab_60_80['clustering_index_network_km'].mean())
print("the mean clustering_index_network_km for subgroup 90-100% is:", com_df_collab_90_100['clustering_index_network_km'].mean())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.boxplot(
    [com_df_collab_40_50['mean_receiver_dist_to_depot_euclidean_km'],
     com_df_collab_60_80['mean_receiver_dist_to_depot_euclidean_km'],
     com_df_collab_90_100['mean_receiver_dist_to_depot_euclidean_km']],
    labels=['40-60%', '70-80%', '90-100%'],
    patch_artist=True,
    boxprops=dict(facecolor='#8dadc3', color='black', alpha=0.7),
    medianprops=dict(color='red', linewidth=2),
    whiskerprops=dict(color='black'),
    capprops=dict(color='black'),
    flierprops=dict(marker='o', markerfacecolor='gray', markersize=5, alpha=0.6)
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.boxplot(
    [com_df_collab_40_50['clustering_index_euclidean_km'],
     com_df_collab_60_80['clustering_index_euclidean_km'],
     com_df_collab_90_100['clustering_index_euclidean_km']],
    labels=['40-60%', '70-80%', '90-100%'],
    patch_artist=True,
    boxprops=dict(facecolor='#8dadc3', color='black', alpha=0.7),
    medianprops=dict(color='red', linewidth=2),
    whiskerprops=dict(color='black'),
    capprops=dict(color='black'),
    flierprops=dict(marker='o', markerfacecolor='gray', markersize=5, alpha=0.6)
)

#### Read and agg NNI metric

In [ ]:
'''read all geo data from clean folder and aggregate NNI metrics'''
nni_df = agg_anls.aggregate_nni_from_clean_folder(
    str(clean_data_path),
    study_area_by_distribution={
        'CLUSTERED': 7000*7000,
        'DISPERSED': 5000*5000,
        'FULLY_RANDOM': 6000 * 6000,
    }
)
nni_df.head(10)

In [ ]:
outside_random_20_af80_nni_df = nni_df[
    (nni_df['depot_location'] == DepotLocation.OUTSIDE.value) &
    (nni_df['receiver_distribution'] == ReceiverDistribution.RANDOM.value) &
    (nni_df['penalty'] == 0.0056) &
    (nni_df['allocation_factor'] == 0.8)
]
# merge with all_metrics_outside_random_20euro_df to get collaboration rate, etc metrics
outside_random_20_af80_nni_df = pd.merge(
    outside_random_20_af80_nni_df,
    all_metrics_outside_random_20euro_af80_df,
    left_on=['instance_id', 'penalty', 'depot_location', 'receiver_distribution', 'allocation_factor'],
    right_on=['instance', 'penalty', 'depot_location', 'receiver_distribution', 'allocation_factor'],
    how='left'
)
outside_random_20_af80_nni_df

In [ ]:
outside_random_20_af80_nni_df.plot.scatter(x='nni',
                                      y='collaboration_rate')

In [ ]:
# Box plot NNI by collaboration_rate subgroups
bins = [0, 0.39, 0.51, 0.8, 1.1]
labels = ['0-0.39', '0.40-0.61', '0.62-0.79', '0.80-1.0']
outside_random_20_af80_nni_df['collab_group'] = pd.cut(
    outside_random_20_af80_nni_df['collaboration_rate'], 
    bins=bins, 
    labels=labels,
    include_lowest=True
)

fig, ax = plt.subplots(figsize=(10, 6))
outside_random_20_af80_nni_df.boxplot(column='nni', by='collab_group', ax=ax)
ax.set_xlabel('Collaboration Rate Group')
ax.set_ylabel('NNI (Nearest Neighbor Index)')
ax.set_title('NNI Distribution by Collaboration Rate Subgroups')
plt.suptitle('')  # Remove automatic title
plt.tight_layout()
plt.show()

# Spatial sensitivity
For this part, we probably need to 
1. first plot the box plots for penalty-sweep and af-sweep across scenarios, to demonstrate/prove the patterns are consistent to the above **cost-benefit sensitivity**.
2. Then, dive into one particular AF-Pen setting group (e.g., af=0.6, pen=0.0056; be consistent with the previous one?). To mainly explore the correlations between spatial index and other key metrics (e.g., total cost savings)
3. Since the clustering index looks irrelevant (little correlated) to other metrics, it is necessary to find another way to demonstrate the power of receivers' spatial distributions.

In [ ]:
ins_spatial_index_df = agg_anls.aggregate_nni_from_clean_folder(
    clean_folder_path=OUTPUT_PATH,
    study_area_by_distribution={
        ReceiverDistribution.CLUSTERED.value: 5000*5000,
        ReceiverDistribution.RANDOM.value: 5000*5000,
        ReceiverDistribution.DISPERSED.value: 9000*9000
    },
    network_node_df=network_nodes,
    network_link_df=network_links,
    network_graph=network_graph,
    verbose=True
)
ins_spatial_index_df

In [ ]:
specific_anls_af = 0.6
specific_anls_penalty = 0.0056

In [ ]:
ins_center_clustered_specific_anls_df = all_metrics_df[
    (all_metrics_df['depot_location'] == DepotLocation.INSIDE.value) &
    (all_metrics_df['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value) &
    (all_metrics_df['allocation_factor'] == specific_anls_af) &
    (all_metrics_df['penalty'] == specific_anls_penalty)
    ]

ins_center_dispersed_specific_anls_df = all_metrics_df[
    (all_metrics_df['depot_location'] == DepotLocation.INSIDE.value) &
    (all_metrics_df['receiver_distribution'] == ReceiverDistribution.DISPERSED.value) &
    (all_metrics_df['allocation_factor'] == specific_anls_af) &
    (all_metrics_df['penalty'] == specific_anls_penalty)
    ]

ins_outside_clustered_specific_anls_df = all_metrics_df[
    (all_metrics_df['depot_location'] == DepotLocation.OUTSIDE.value) &
    (all_metrics_df['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value) &
    (all_metrics_df['allocation_factor'] == specific_anls_af) &
    (all_metrics_df['penalty'] == specific_anls_penalty)
    ]

ins_outside_dispersed_specific_anls_df = all_metrics_df[
    (all_metrics_df['depot_location'] == DepotLocation.OUTSIDE.value) &
    (all_metrics_df['receiver_distribution'] == ReceiverDistribution.DISPERSED.value) &
    (all_metrics_df['allocation_factor'] == specific_anls_af) &
    (all_metrics_df['penalty'] == specific_anls_penalty)
    ]


In [ ]:
ins_center_clustered_specific_anls_df

In [ ]:
spatial_cols = ['depot_location', 'receiver_distribution', 'allocation_factor', 'penalty', 'instance_id',
                'nni', 'pattern', 'centroid_to_depot_euclidean_km', 'centroid_to_depot_network_km']

ins_center_clustered_specific_anls_df = pd.merge(
    ins_center_clustered_specific_anls_df,
    ins_spatial_index_df[spatial_cols],
    left_on=['depot_location', 'receiver_distribution', 'allocation_factor', 'penalty', 'instance'],
    right_on=['depot_location', 'receiver_distribution', 'allocation_factor', 'penalty', 'instance_id'],
    how='left'
)

ins_center_dispersed_specific_anls_df = pd.merge(
    ins_center_dispersed_specific_anls_df,
    ins_spatial_index_df[spatial_cols],
    left_on=['depot_location', 'receiver_distribution', 'allocation_factor', 'penalty', 'instance'],
    right_on=['depot_location', 'receiver_distribution', 'allocation_factor', 'penalty', 'instance_id'],
    how='left'
)

ins_outside_clustered_specific_anls_df = pd.merge(
    ins_outside_clustered_specific_anls_df,
    ins_spatial_index_df[spatial_cols],
    left_on=['depot_location', 'receiver_distribution', 'allocation_factor', 'penalty', 'instance'],
    right_on=['depot_location', 'receiver_distribution', 'allocation_factor', 'penalty', 'instance_id'],
    how='left'
)

ins_outside_dispersed_specific_anls_df = pd.merge(
    ins_outside_dispersed_specific_anls_df,
    ins_spatial_index_df[spatial_cols],
    left_on=['depot_location', 'receiver_distribution', 'allocation_factor', 'penalty', 'instance'],
    right_on=['depot_location', 'receiver_distribution', 'allocation_factor', 'penalty', 'instance_id'],
    how='left'
)

### Box plots of four scenarios' metrics


#### Collab. rate (center-clustered-af0.6)

In [ ]:
figure_plot.box_plot(
    data_list=all_metrics_center_clustered_df.query("allocation_factor == 0.6"),
    cat_col='penalty_std',
    col_name='collaboration_rate',
    box_colors=["#83A7BE"] * len(PENALTY_LIST),
    # box_colors=sns.color_palette("Blues", n_colors=len(PENALTY_LIST)),
    box_alpha=0.9,
    se_box_color_for_lines=True,
     # Display options
    flier_marker='X',
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c4596e",
    median_linewidth=1,
    show_mean=True,
    mean_size=5,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Collaboration rate',
    xlabel='Cost of collaboration (euros/hour)',
    #--- Violin plot ---
    show_violin=False,
    violin_alpha=0.3, 
    violin_width=0.9,
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(4.2, 3.6),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='collaboration_rate_by_penalty_af60_all_center_clustered_receivers.png',
)

#### Collab. rate (center-dispersed-af0.6)

In [ ]:
figure_plot.box_plot(
    data_list=all_metrics_center_dispersed_df.query("allocation_factor == 0.6"),
    cat_col='penalty_std',
    col_name='collaboration_rate',
    box_colors=["#83A7BE"] * len(PENALTY_LIST),
    # box_colors=sns.color_palette("Blues", n_colors=len(PENALTY_LIST)),
    box_alpha=0.9,
    se_box_color_for_lines=True,
     # Display options
    flier_marker='X',
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c4596e",
    median_linewidth=1,
    show_mean=True,
    mean_size=5,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Collaboration rate',
    xlabel='Cost of collaboration (euros/hour)',
    #--- Violin plot ---
    show_violin=False,
    violin_alpha=0.3, 
    violin_width=0.9,
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(4.2, 3.6),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='collaboration_rate_by_penalty_af60_all_center_dispersed_receivers.png',
)

#### Collab. rate (outside-clustered-af0.6)

In [ ]:
figure_plot.box_plot(
    data_list=all_metrics_outside_clustered_df.query("allocation_factor == 0.6"),
    cat_col='penalty_std',
    col_name='collaboration_rate',
    box_colors=["#83A7BE"] * len(PENALTY_LIST),
    # box_colors=sns.color_palette("Blues", n_colors=len(PENALTY_LIST)),
    box_alpha=0.9,
    se_box_color_for_lines=True,
     # Display options
    flier_marker='X',
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c4596e",
    median_linewidth=1,
    show_mean=True,
    mean_size=5,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Collaboration rate',
    xlabel='Cost of collaboration (euros/hour)',
    #--- Violin plot ---
    show_violin=False,
    violin_alpha=0.3, 
    violin_width=0.9,
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(4.2, 3.6),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='collaboration_rate_by_penalty_af60_all_outside_clustered_receivers.png',
)

#### Collab. rate (outside-dispersed-af0.6)

In [ ]:
all_metrics_outside_dispersed_df.query("allocation_factor == 0.6")

In [ ]:
ins_outside_dispersed_specific_anls_df

In [ ]:
figure_plot.box_plot(
    data_list=all_metrics_outside_dispersed_df.query("allocation_factor == 0.6"),
    cat_col='penalty_std',
    col_name='collaboration_rate',
    box_colors=["#83A7BE"] * len(PENALTY_LIST),
    # box_colors=sns.color_palette("Blues", n_colors=len(PENALTY_LIST)),
    box_alpha=0.9,
    se_box_color_for_lines=True,
     # Display options
    flier_marker='X',
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c4596e",
    median_linewidth=1,
    show_mean=True,
    mean_size=5,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Collaboration rate',
    xlabel='Cost of collaboration (euros/hour)',
    #--- Violin plot ---
    show_violin=False,
    violin_alpha=0.3, 
    violin_width=0.9,
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(4.2, 3.6),
    dpi=350,
    figure_folder=OUTPUT_FIG_PATH,
    filename='collaboration_rate_by_penalty_af60_all_outside_dispersed_receivers.png',
)

### Trial on Moran's I and Getis Ord 

In [ ]:
link_data_path = repo_root / 'data/freightChessboardRC/cleanMoreDispersed'
all_stat_collab_links, all_stat_collab_links_agg = agg_anls.aggregate_collaborative_receivers_by_link(
    str(link_data_path),
    verbose=True
)

In [ ]:
ins_outside_dispersed_specific_stat_collab_links = all_stat_collab_links[
    (all_stat_collab_links['allocation_factor'] == specific_anls_af) &
    (all_stat_collab_links['penalty'] == specific_anls_penalty) &
    (all_stat_collab_links['depot_location'] == DepotLocation.OUTSIDE.value) &
    (all_stat_collab_links['receiver_distribution'] == ReceiverDistribution.DISPERSED.value)
]

ins_outside_clustered_specific_stat_collab_links = all_stat_collab_links[
    (all_stat_collab_links['allocation_factor'] == specific_anls_af) &
    (all_stat_collab_links['penalty'] == specific_anls_penalty) &
    (all_stat_collab_links['depot_location'] == DepotLocation.OUTSIDE.value) &
    (all_stat_collab_links['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value)
]

ins_center_dispersed_specific_stat_collab_links = all_stat_collab_links[
    (all_stat_collab_links['allocation_factor'] == specific_anls_af) &
    (all_stat_collab_links['penalty'] == specific_anls_penalty) &
    (all_stat_collab_links['depot_location'] == DepotLocation.INSIDE.value) &
    (all_stat_collab_links['receiver_distribution'] == ReceiverDistribution.DISPERSED.value)
]

ins_center_clustered_specific_stat_collab_links = all_stat_collab_links[
    (all_stat_collab_links['allocation_factor'] == specific_anls_af) &
    (all_stat_collab_links['penalty'] == specific_anls_penalty) &
    (all_stat_collab_links['depot_location'] == DepotLocation.INSIDE.value) &
    (all_stat_collab_links['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value)
]

In [ ]:
ins_center_specific_stat_collab_links = all_stat_collab_links[
    (all_stat_collab_links['allocation_factor'] == specific_anls_af) &
    (all_stat_collab_links['penalty'] == specific_anls_penalty) &
    (all_stat_collab_links['depot_location'] == DepotLocation.INSIDE.value)
]

ins_outside_specific_stat_collab_links = all_stat_collab_links[
    (all_stat_collab_links['allocation_factor'] == specific_anls_af) &
    (all_stat_collab_links['penalty'] == specific_anls_penalty) &
    (all_stat_collab_links['depot_location'] == DepotLocation.OUTSIDE.value)
]

In [ ]:
mean_collab_links_center_clustered = spatial_anls.compute_mean_collab_count_by_link(
    ins_center_clustered_specific_stat_collab_links,
    central_area_network_gdf,)
mean_collab_links_center_dispersed = spatial_anls.compute_mean_collab_count_by_link(
    ins_center_dispersed_specific_stat_collab_links,
    full_network_gdf,)
mean_collab_links_outside_clustered = spatial_anls.compute_mean_collab_count_by_link(
    ins_outside_clustered_specific_stat_collab_links,
    central_area_network_gdf,)
mean_collab_links_outside_dispersed = spatial_anls.compute_mean_collab_count_by_link(
    ins_outside_dispersed_specific_stat_collab_links,
    full_network_gdf,)

mean_collab_links_center = spatial_anls.compute_mean_collab_count_by_link(
    ins_center_specific_stat_collab_links,
    central_area_network_gdf,)
mean_collab_links_outside = spatial_anls.compute_mean_collab_count_by_link(
    ins_outside_specific_stat_collab_links,
    full_network_gdf,) 

In [ ]:
mean_collab_links_center.plot(column='total_collab_count',
                                        cmap='Blues',
                                        legend=True,
                                        figsize=(8, 6),
                                        # title='Mean Collaborative Receivers per Link\n(Center - Clustered)',
                                        # edgecolor='k',
                                        linewidth=3)

In [ ]:
mean_collab_links_outside.plot(column='total_collab_count',
                                        cmap='Blues',
                                        legend=True,
                                        figsize=(8, 6),
                                        # title='Mean Collaborative Receivers per Link\n(Center - Clustered)',
                                        # edgecolor='k',
                                        linewidth=3)

In [ ]:
_result = spatial_anls.compute_getis_ord_statistics(
    mean_collab_links_outside, 
    col_name='total_collab_count',
    # weights_type='knn',
    # k_neighbors=5
    weights_type='queen',
    # distance_threshold=1500,
    # binary=True
)

In [ ]:
spatial_anls.compute_getis_ord_statistics(
    mean_collab_links_center, 
    col_name='total_collab_count',
    # weights_type='knn',
    # k_neighbors=5
    weights_type='queen',
    # distance_threshold=1500,
    # binary=True
)

In [ ]:
result_moran_ = spatial_anls.compute_moran_statistics(
    mean_collab_links_center,
     weights_type='queen', 
    col_name='total_collab_count'
)

In [ ]:
result_moran_ = spatial_anls.compute_moran_statistics(
    mean_collab_links_outside,
     weights_type='queen', 
    col_name='total_collab_count'
)

### Map and locations

In [ ]:
_sel_ins = 6
dispersed_ins_receiver_df = gpd.read_file(os.path.join(
    OUTPUT_PATH, 
    f'{DepotLocation.OUTSIDE.value}-{ReceiverDistribution.DISPERSED.value}-af{ALLOCATION_FACTOR:.02f}-p{0.0056}-i{_sel_ins:02d}',
    'geo_data.geojson'))
clustered_ins_receiver_df = gpd.read_file(os.path.join(
    OUTPUT_PATH,
    f'{DepotLocation.OUTSIDE.value}-{ReceiverDistribution.CLUSTERED.value}-af{ALLOCATION_FACTOR:.02f}-p{0.0056}-i{_sel_ins:02d}',
    'geo_data.geojson'))

In [ ]:
''' Plot the dispersed scenario receiver locations at link midpoints '''
figure_plot.network_locations_plot(
    network=network,
    locations_gdf=dispersed_ins_receiver_df,
    # Links settings
    show_links=True,
    show_nodes=True,
    use_arrows=True,
    link_color="#878A8B",
    link_alpha=0.6,
    link_linewidth=0.6,
    arrow_scale=12,
    arrow_shrink=5,
    # Nodes settings
    node_size=15,
    node_color="#A7A7A7",
    node_alpha=0.8,
    # 使用link中点定位 (新功能)
    use_link_midpoint=True,
    link_id_col='link_id',
    # Central region
    show_central_region=False,
    central_region_bounds=(2000, 2000, 5000, 5000),
    central_region_label='Central Region (2000-7000)',
    #---X/Y labels---
    xlabel='X coordinate (m)',
    ylabel='Y coordinate (m)',
    label_size=14,

    # Axis limits
    xlim=(-500, 9500),
    ylim=(-500, 9500),
    #---Title and style---
    # title='Receiver Locations at Link Midpoints (Dispersed)',
    figure_size=(4, 4),
    dpi=350,
    show_grid=True,
    show_legend=False,
    #---Output---
    figure_folder=OUTPUT_FIG_PATH,
    filename=f'receiver_locations_dispersed_outside_random_af{specific_anls_af:.02f}_p{specific_anls_penalty:.04f}_i{_sel_ins:02d}.png'
)

In [ ]:
''' Plot the clustered scenario receiver locations at link midpoints '''
figure_plot.network_locations_plot(
    network=network,
    locations_gdf=clustered_ins_receiver_df,
    # Links settings
    show_links=True,
    show_nodes=True,
    use_arrows=True,
    link_color="#878A8B",
    link_alpha=0.6,
    link_linewidth=0.6,
    arrow_scale=12,
    arrow_shrink=5,
    # Nodes settings
    node_size=15,
    node_color="#A7A7A7",
    node_alpha=0.8,
    # 使用link中点定位 (新功能)
    use_link_midpoint=True,
    link_id_col='link_id',
    # Central region
    show_central_region=True,
    central_region_bounds=(2000, 2000, 5000, 5000),
    central_region_label='Central Region (2000-7000)',
    #---X/Y labels---
    xlabel='X coordinate (m)',
    ylabel='Y coordinate (m)',
    label_size=14,

    # Axis limits
    xlim=(-500, 9500),
    ylim=(-500, 9500),
    #---Title and style---
    # title='Receiver Locations at Link Midpoints (Clustered)',
    figure_size=(4, 4),
    dpi=350,
    show_grid=True,
    show_legend=False,
    #---Output---
    figure_folder=OUTPUT_FIG_PATH,
    filename=f'receiver_locations_clustered_outside_random_af{specific_anls_af:.02f}_p{specific_anls_penalty:.04f}_i{_sel_ins:02d}.png'
)

### Collaboration rate

In [ ]:
''' Box plot using figure_plot module '''
del_ins = (-1, -50)
# Example usage with the new box_plot function
fig, ax, bp = figure_plot.box_plot(
    data_list=[
        ins_center_clustered_specific_anls_df[~ins_center_clustered_specific_anls_df['instance'].between(del_ins[0], del_ins[1])],
        ins_center_dispersed_specific_anls_df[~ins_center_dispersed_specific_anls_df['instance'].between(del_ins[0], del_ins[1])],
        ins_outside_clustered_specific_anls_df[~ins_outside_clustered_specific_anls_df['instance'].between(del_ins[0], del_ins[1])],
        ins_outside_dispersed_specific_anls_df[~ins_outside_dispersed_specific_anls_df['instance'].between(del_ins[0], del_ins[1])]
    ],
    col_name='collaboration_rate',
    labels=['Center-Clustered', 'Center-Dispersed', 'Outside-Clustered', 'Outside-Dispersed'],
    # Box colors
    box_colors=["#89ADA8", "#ddd2e9c4", "#345e55", "#8e76a6"],
    # Background colors for each box region
    bg_colors=["#CBE6D9", "#f7e0fb", "#9ac99a", "#d9b3ed"],
    # Display options
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c3435d",
    show_mean=True,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Collaboration rate',
    # title=f'Collaboration Rate Distribution (AF={specific_anls_af}, Penalty={specific_anls_penalty})',
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(6, 2.7),
    figure_folder=OUTPUT_FIG_PATH,
    # filename=f'boxplot_collaboration_rate_across_spatial_distributions_scenarios_af{specific_anls_af:.02f}_p{specific_anls_penalty:.04f}_i{_sel_ins:02d}.png'
)

It seems not necessary to compare VKT, TT and TKT, 
since the scenarios with the outside depot have way higher values than others.

### VKT

In [ ]:
''' Box plot '''
fig, ax, bp = figure_plot.box_plot(
    data_list=[
        ins_center_clustered_specific_anls_df,
        ins_outside_clustered_specific_anls_df,
        ins_center_dispersed_specific_anls_df,
        ins_outside_dispersed_specific_anls_df
    ],
    col_name='VKT_km',
    labels=['Center-Clustered', 'Outside-Clustered', 'Center-Dispersed', 'Outside-Dispersed'],
    # Box colors
    box_colors=["#89ADA8",  "#345e55", "#ddd2e9c4", "#8e76a6"],
    # Background colors for each box region
    bg_colors=["#CBE6D9", "#9ac99a", "#f7e0fb",  "#d9b3ed"],
    # Display options
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c3435d",
    show_mean=True,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='VKT',
    # title=f'Collaboration Rate Distribution (AF={specific_anls_af}, Penalty={specific_anls_penalty})',
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(6, 2.7),
    figure_folder=OUTPUT_FIG_PATH,
    #filename=f'boxplot_vkt_across_spatial_distributions_scenarios_af_{specific_anls_af}.png'
)

In [ ]:
''' Box plot '''
fig, ax, bp = figure_plot.box_plot(
    data_list=[
        ins_center_clustered_specific_anls_df,
        ins_outside_clustered_specific_anls_df,
        ins_center_dispersed_specific_anls_df,
        ins_outside_dispersed_specific_anls_df
    ],
    col_name='VKT_km',
    labels=['Center-Clustered', 'Outside-Clustered', 'Center-Dispersed', 'Outside-Dispersed'],
    # Box colors
    box_colors=["#83A7BE"] * 4,
    # Background colors for each box region
    #bg_colors=["#CBE6D9", "#9ac99a", "#f7e0fb",  "#d9b3ed"],
    # Display options
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c3435d",
    show_mean=True,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='VKT',
    # title=f'Collaboration Rate Distribution (AF={specific_anls_af}, Penalty={specific_anls_penalty})',
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(6, 2.7),
    figure_folder=OUTPUT_FIG_PATH,
    #filename=f'boxplot_vkt_across_spatial_distributions_scenarios_af_{specific_anls_af}.png'
)

### VTT

In [ ]:
''' Box plot '''
fig, ax = plt.subplots(figsize=(10, 6))
ax.boxplot(
    [ins_center_clustered_specific_anls_df['VTT_seconds'],
     ins_center_dispersed_specific_anls_df['VTT_seconds'],
     ins_outside_clustered_specific_anls_df['VTT_seconds'],
     ins_outside_dispersed_specific_anls_df['VTT_seconds']],
    labels=['Center-Clustered', 'Center-Dispersed', 'Outside-Clustered', 'Outside-Dispersed'],
    patch_artist=True,
    boxprops=dict(facecolor='#8dadc3', color='black', alpha=0.7),
    medianprops=dict(color='red', linewidth=2),
    whiskerprops=dict(color='black'),
    capprops=dict(color='black'),
    flierprops=dict(marker='o', markerfacecolor='gray', markersize=5, alpha=0.6)
)
ax.set_title(f'VTT Distribution (AF={specific_anls_af}, Penalty={specific_anls_penalty})')
ax.set_ylabel('VTT (seconds)')
plt.tight_layout()

plt.show()

### TKT

In [ ]:
''' Box plot '''
_ins_center_clustered_specific_anls_df = ins_center_clustered_specific_anls_df.copy()
_ins_center_clustered_specific_anls_df['TKT_tonkm'] = _ins_center_clustered_specific_anls_df['TKT_tonkm'] / 1000
_ins_center_dispersed_specific_anls_df = ins_center_dispersed_specific_anls_df.copy()
_ins_center_dispersed_specific_anls_df['TKT_tonkm'] = _ins_center_dispersed_specific_anls_df['TKT_tonkm'] / 1000
_ins_outside_clustered_specific_anls_df = ins_outside_clustered_specific_anls_df.copy()
_ins_outside_clustered_specific_anls_df['TKT_tonkm'] = _ins_outside_clustered_specific_anls_df['TKT_tonkm'] / 1000
_ins_outside_dispersed_specific_anls_df = ins_outside_dispersed_specific_anls_df.copy()
_ins_outside_dispersed_specific_anls_df['TKT_tonkm'] = _ins_outside_dispersed_specific_anls_df['TKT_tonkm'] / 1000

fig, ax, bp = figure_plot.box_plot(
    data_list=[
        _ins_center_clustered_specific_anls_df,
        _ins_center_dispersed_specific_anls_df,
        _ins_outside_clustered_specific_anls_df,
        _ins_outside_dispersed_specific_anls_df
    ],
    col_name='TKT_tonkm',
    labels=['Center-Clustered', 'Center-Dispersed', 'Outside-Clustered', 'Outside-Dispersed'],
    # Box colors
    box_colors=["#89ADA8", "#ddd2e9c4", "#345e55", "#8e76a6"],
    # Background colors for each box region
    bg_colors=["#CBE6D9", "#f7e0fb", "#9ac99a", "#d9b3ed"],
    # Display options
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c3435d",
    show_mean=True,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Ton-km travelled',
    # title=f'Collaboration Rate Distribution (AF={specific_anls_af}, Penalty={specific_anls_penalty})',
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(6, 2.7),
    figure_folder=OUTPUT_FIG_PATH,
    filename=f'boxplot_tkt_across_spatial_distributions_scenarios_af_{specific_anls_af}.png'
)

In [ ]:
''' Box plot '''
_ins_center_clustered_specific_anls_df = ins_center_clustered_specific_anls_df.copy()
_ins_center_clustered_specific_anls_df['TKT_tonkm'] = _ins_center_clustered_specific_anls_df['TKT_tonkm'] / 1000
_ins_center_dispersed_specific_anls_df = ins_center_dispersed_specific_anls_df.copy()
_ins_center_dispersed_specific_anls_df['TKT_tonkm'] = _ins_center_dispersed_specific_anls_df['TKT_tonkm'] / 1000
_ins_outside_clustered_specific_anls_df = ins_outside_clustered_specific_anls_df.copy()
_ins_outside_clustered_specific_anls_df['TKT_tonkm'] = _ins_outside_clustered_specific_anls_df['TKT_tonkm'] / 1000
_ins_outside_dispersed_specific_anls_df = ins_outside_dispersed_specific_anls_df.copy()
_ins_outside_dispersed_specific_anls_df['TKT_tonkm'] = _ins_outside_dispersed_specific_anls_df['TKT_tonkm'] / 1000

fig, ax, bp = figure_plot.box_plot(
    data_list=[
        _ins_center_clustered_specific_anls_df,
        _ins_outside_clustered_specific_anls_df,
        _ins_center_dispersed_specific_anls_df,
        _ins_outside_dispersed_specific_anls_df
    ],
    col_name='TKT_tonkm',
    labels=['Center-Clustered',  'Outside-Clustered', 'Center-Dispersed', 'Outside-Dispersed'],
    # Box colors
    box_colors=["#83A7BE"] * 4,
    # Background colors for each box region
    #bg_colors=["#CBE6D9", "#f7e0fb", "#9ac99a", "#d9b3ed"],
    # Display options
    show_scatter=False,
    scatter_use_box_color=True,
    scatter_alpha=0.4,
    median_color="#c3435d",
    show_mean=True,
    mean_color="#f1bb64",
    use_box_color_for_lines=True,
    # Labels
    ylabel='Ton-km travelled',
    # title=f'Collaboration Rate Distribution (AF={specific_anls_af}, Penalty={specific_anls_penalty})',
    # Grid
    show_grid=True,
    grid_axis='y',
    #---output---
    label_size=13,
    figure_size=(6, 2.7),
    figure_folder=OUTPUT_FIG_PATH,
    #filename=f'boxplot_tkt_across_spatial_distributions_scenarios_af_{specific_anls_af}.png'
)

### Fleet size

In [ ]:
fleet_size_data_dict = {
    'Center-Clustered': ins_center_clustered_specific_anls_df,
    'Center-Dispersed': ins_center_dispersed_specific_anls_df,
    'Outside-Clustered': ins_outside_clustered_specific_anls_df,
    'Outside-Dispersed': ins_outside_dispersed_specific_anls_df
}

In [ ]:
fig, ax = figure_plot.quadrant_donut_chart(
    data_dict=fleet_size_data_dict,
    value_col='final_fleet_size',
    hole_radius=0.4,
    xlabel_right='Dispersed →',
    xlabel_left='← Clustered',
    ylabel_top='↑ Center',
    ylabel_bottom='↓ Outside',
    explode=0.03,
    dpi=350,
    figsize=(6, 6),
)

In [ ]:


fig, ax = figure_plot.nested_donut_chart(
    data_dict,
    group_col='final_fleet_size',
    # title='Fleet Size Distribution by Scenario',
    inner_colors=["#88ABA7", "#ddd2e9c4", "#4d8075", "#b7a4ca"],
    outer_color_list=["#dcbdc3", "#c18d97", "#844954CF", "#844954"],
    outer_cmap='coolwarm',
    show_inner_labels=False,
    show_inner_pct=False,
    show_outer_count=False,
    show_outer_labels=False,
    hole_radius=0.4,
    inner_label_size=5,
    dpi=350,
    outer_label_size=10,
    figsize=(5, 5),
    show_legend=False,
    figure_folder=OUTPUT_FIG_PATH,
    filename='nested_donut_fleet_size_by_scenario.png',
     transparent_bg=True
)

### Spatial index
Relations: (mean-dist-to-depot; clustering index; ) <-> (VKT, TT, TKT, Cost savings, scores, etc)

In [ ]:
spatial_concat_df = pd.concat([
    ins_center_clustered_specific_anls_df,
    ins_center_dispersed_specific_anls_df,
    ins_outside_clustered_specific_anls_df,
    ins_outside_dispersed_specific_anls_df
], ignore_index=True)

In [ ]:
spatial_concat_df['class'] = spatial_concat_df.apply(lambda row: f"{row['depot_location']}_{row['receiver_distribution']}", axis=1)
spatial_concat_df

In [ ]:
spatial_concat_df.groupby('class')['clustering_index_network_km'].mean()

In [ ]:
spatial_concat_df.groupby('class')['clustering_index_euclidean_km'].mean()

In [ ]:
spatial_concat_df.groupby('class')['mean_receiver_dist_to_depot_network_km'].mean()

In [ ]:
spatial_concat_df.groupby('class')['mean_receiver_dist_to_depot_euclidean_km'].mean()

In [ ]:

figure_plot.joint_scatter_plot(
    data_df=spatial_concat_df,
    x_col='mean_receiver_dist_to_depot_euclidean_km',
    y_col='nni',
    # size_group_col='total_cost_savings',
    size_group_col='collaboration_rate',
    color_group_col='class',
    show_size_legend=False,

)

In [ ]:
figure_plot.scatter_regression_plot(
    x_col='clustering_index_euclidean_km',
    y_col='total_cost_savings',
    data_df=spatial_concat_df,
    figure_size=(5.8,4),
    xlabel='Disperse index (km)',
    ylabel='Total cost savings',
    label_size=14,
    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
    show_corr=True,
    add_regression=True,
    show_equation=False,
    show_r2=False,
    scatter_color='#458CBC',
    scatter_alpha=0.9,
    reg_color='#0b2c60',
    # Output
    # figure_folder=OUTPUT_FIG_PATH,
    # filename='scatter_regression_cost_savings_vs_clustering_index.png'
    ) 

In [ ]:
figure_plot.scatter_regression_plot(
    x_col='mean_receiver_dist_to_depot_euclidean_km',
    y_col='total_cost_savings',
    data_df=spatial_concat_df,
    figure_size=(5.8,4),
    xlabel='Mean receiver distance to depot (km)',
    ylabel='Total cost savings',
    label_size=14,
    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
    show_corr=True,
    add_regression=True,
    show_equation=False,
    show_r2=True,
    annotation_fontsize=12,
    scatter_color='#458CBC',
    scatter_alpha=0.9,
    reg_color='#0b2c60',
    # Output
    figure_folder=OUTPUT_FIG_PATH,
    #filename='scatter_regression_cost_savings_vs_mean_receiver_dist2Depot.png'
    ) 

In [ ]:
figure_plot.scatter_regression_plot(
    x_col='mean_receiver_dist_to_depot_network_km',
    y_col='total_cost_savings',
    data_df=spatial_concat_df,
    figure_size=(5.8,4),
    xlabel='Mean receiver distance to depot (km)',
    ylabel='Total cost savings',
    label_size=14,
    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
    show_corr=True,
    add_regression=True,
    show_equation=False,
    show_r2=True,
    annotation_fontsize=12,
    scatter_color='#458CBC',
    scatter_alpha=0.9,
    reg_color='#0b2c60',
    # Output
    figure_folder=OUTPUT_FIG_PATH,
    #filename='scatter_regression_cost_savings_vs_mean_receiver_dist2Depot.png'
    ) 

In [ ]:
figure_plot.scatter_regression_plot(
    x_col='nni',
    y_col='total_cost_savings',
    data_df=spatial_concat_df[spatial_concat_df['pattern'] != 'random'],
    figure_size=(5.8,4),
    xlabel='Nearest Neighbor Index (NNI)',
    ylabel='Total cost savings',
    label_size=14,
    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
    show_corr=True,
    add_regression=True,
    show_equation=False,
    show_r2=True,
    annotation_fontsize=12,
    scatter_color='#458CBC',
    scatter_alpha=0.9,
    reg_color='#0b2c60',
    # Output
    figure_folder=OUTPUT_FIG_PATH,
    #filename='scatter_regression_cost_savings_vs_nni.png'
    ) 

In [ ]:
figure_plot.scatter_regression_plot(
    x_col='centroid_to_depot_euclidean_km',
    y_col='total_cost_savings',
    data_df=spatial_concat_df,
    figure_size=(5.8,4),
    xlabel='Centroid to depot distance (km)',
    ylabel='Total cost savings',
    label_size=14,
    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
    show_corr=True,
    add_regression=True,
    show_equation=False,
    show_r2=True,
    annotation_fontsize=12,
    scatter_color='#458CBC',
    scatter_alpha=0.9,
    reg_color='#0b2c60',
    # Output
    figure_folder=OUTPUT_FIG_PATH,
    #filename='scatter_regression_cost_savings_vs_centroid_to_depot.png'
    ) 

In [ ]:
figure_plot.scatter_regression_plot(
    x_col='centroid_to_depot_euclidean_km',
    y_col='total_cost_savings',
    data_df=spatial_concat_df[spatial_concat_df['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value],
    figure_size=(5.8,4),
    xlabel='Centroid to depot distance (km)',
    ylabel='Total cost savings',
    label_size=14,
    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
    show_corr=True,
    add_regression=True,
    show_equation=False,
    show_r2=True,
    annotation_fontsize=12,
    scatter_color='#458CBC',
    scatter_alpha=0.9,
    reg_color='#0b2c60',
    # Output
    figure_folder=OUTPUT_FIG_PATH,
    #filename='scatter_regression_cost_savings_vs_centroid_to_depot.png'
    ) 

In [ ]:
figure_plot.scatter_regression_plot(
    x_col='centroid_to_depot_euclidean_km',
    y_col='total_cost_savings',
    data_df=spatial_concat_df[spatial_concat_df['receiver_distribution'] == ReceiverDistribution.DISPERSED.value],
    figure_size=(5.8,4),
    xlabel='Centroid to depot distance (km)',
    ylabel='Total cost savings',
    label_size=14,
    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
    show_corr=True,
    add_regression=True,
    show_equation=False,
    show_r2=True,
    annotation_fontsize=12,
    scatter_color='#458CBC',
    scatter_alpha=0.9,
    reg_color='#0b2c60',
    # Output
    figure_folder=OUTPUT_FIG_PATH,
    #filename='scatter_regression_cost_savings_vs_centroid_to_depot.png'
    ) 

In [ ]:
figure_plot.scatter_regression_plot(
    x_col='centroid_to_depot_network_km',
    y_col='total_cost_savings',
    data_df=spatial_concat_df,
    figure_size=(5,3),
    xlabel='Centroid to depot network distance (km)',
    ylabel='Total cost savings',
    label_size=14,
    y_tick_step=100,
    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
    show_corr=True,
    add_regression=True,
    show_equation=False,
    show_r2=True,
    annotation_fontsize=12,
    scatter_color='#458CBC',
    scatter_alpha=0.9,
    reg_color='#0b2c60',
    # Output
    figure_folder=OUTPUT_FIG_PATH,
    #filename='scatter_regression_cost_savings_vs_centroid_to_depot_network.png'
    ) 

In [ ]:
figure_plot.scatter_regression_plot(
    x_col='centroid_to_depot_network_km',
    y_col='total_cost_savings',
    data_df=spatial_concat_df[spatial_concat_df['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value],
    figure_size=(5.8,4),
    xlabel='Centroid to depot distance (km)',
    ylabel='Total cost savings',
    label_size=14,
    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
    show_corr=True,
    add_regression=True,
    show_equation=False,
    show_r2=True,
    annotation_fontsize=12,
    scatter_color='#458CBC',
    scatter_alpha=0.9,
    reg_color='#0b2c60',
    # Output
    figure_folder=OUTPUT_FIG_PATH,
    #filename='scatter_regression_cost_savings_vs_centroid_to_depot.png'
    ) 

In [ ]:
figure_plot.scatter_regression_plot(
    x_col='centroid_to_depot_euclidean_km',
    y_col='total_cost_savings',
    data_df=spatial_concat_df[spatial_concat_df['receiver_distribution'] == ReceiverDistribution.DISPERSED.value],
    figure_size=(5.8,4),
    xlabel='Centroid to depot distance (km)',
    ylabel='Total cost savings',
    label_size=14,
    # title='Scatter plot with regression line: Cost Savings vs Collaboration Rate',
    show_corr=True,
    add_regression=True,
    show_equation=False,
    show_r2=True,
    annotation_fontsize=12,
    scatter_color='#458CBC',
    scatter_alpha=0.9,
    reg_color='#0b2c60',
    # Output
    figure_folder=OUTPUT_FIG_PATH,
    #filename='scatter_regression_cost_savings_vs_centroid_to_depot.png'
    ) 